# India Economic Data Observatory — Quickstart

This notebook walks through loading one indicator (GDP) with the pipeline, inspecting its `Metadata`, checking whether the run got live or synthetic data, and plotting it with the metadata caption attached — the "metadata travels with the chart" pattern this project is built around.

Run this from the project root (or adjust the `sys.path.insert` below if you move it).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from data_observatory import loaders, viz
from data_observatory.metadata import DataStatus

## 1. Load an indicator

`loaders.load(key, start, end, allow_live=True)` always attempts the real official endpoint first (World Bank's `NY.GDP.MKTP.CD` for GDP), and falls back to a deterministic synthetic series only if that fails. It returns `(frame, metadata)`.

In [2]:
frame, meta = loaders.load("gdp", "2000-01-01", "2023-01-01", allow_live=True)
frame.tail()

,value,imputed
2019-01-01,3.794651e+12,False
2020-01-01,4.175069e+12,False
2021-01-01,4.558513e+12,False
2022-01-01,4.803746e+12,False
2023-01-01,5.511462e+12,False


## 2. Inspect the metadata

This is the object every chart, table, and downstream notebook cell must check before presenting a number as real.

In [3]:
print(meta.long_description())
print()
print("Is this live/official data?", meta.status is DataStatus.OFFICIAL)

Indicator: GDP (current US$)
Status: SYNTHETIC / ILLUSTRATIVE
Provider / source: World Bank — World Development Indicators — NY.GDP.MKTP.CD
URL: https://api.worldbank.org/v2/country/IND/indicator/NY.GDP.MKTP.CD
Definition: Gross Domestic Product at purchaser's prices, converted to current US dollars.
Unit: current US$
Frequency: annual
Date coverage: 2000-01-01 to 2023-01-01
Transformations applied: none beyond World Bank's own current-US$ conversion
Limitations: Current-US$ GDP mixes real growth with exchange-rate and price-level effects; not adjusted for inflation or PPP. Live fetch unavailable or disabled; showing deterministic synthetic fallback.

Is this live/official data? False


## 3. List all available indicators by category

In [4]:
for category, keys in loaders.list_categories().items():
    print(category)
    for k in keys:
        print("  -", loaders.INDICATORS[k].name)

Growth
  - GDP (current US$)
  - GDP per capita (current US$)
Prices
  - CPI inflation (annual %)
  - WPI inflation (annual %, illustrative subset)
Labour
  - Unemployment rate
  - Labour force participation rate
Government
  - Fiscal balance (cash surplus/deficit, % of GDP)
  - Central government debt (% of GDP)
External sector
  - Exports of goods and services (current US$)
  - Imports of goods and services (current US$)
  - Current account balance (% of GDP)
  - Official exchange rate (INR per US$)
Financial system
  - Policy interest rate (illustrative; RBI repo rate proxy)
  - Bank credit growth (p.p. change, domestic credit to private sector)


## 4. Plot it, with the metadata caption attached

`viz.line_chart` returns a Plotly figure whose caption annotation states the provider, unit, frequency, date coverage, and — crucially — the live/synthetic status, directly on the chart, not just as a separate return value someone could forget to display.

In [5]:
fig = viz.line_chart(frame, meta)
fig.show()

## 5. Compare two indicators on one chart

In [6]:
frame_cpi, meta_cpi = loaders.load("cpi", "2000-01-01", "2023-01-01", allow_live=True)
fig2 = viz.comparison_chart(frame, meta, frame_cpi, meta_cpi)
fig2.show()

## 6. Why the status might say SYNTHETIC here

If both charts above show the amber dashed "SYNTHETIC / ILLUSTRATIVE" style rather than the solid navy "LIVE / OFFICIAL" one, your network (or this sandbox's egress policy) blocked the World Bank API call — not a bug in this pipeline. See `docs/DATA_SOURCES.md` for the exact probe result this project's author got when building it, and try re-running this notebook on a machine with ordinary internet access to see live data flow through the same code path.